<a href="https://colab.research.google.com/github/mikitro04/Simple-AA-Wallet/blob/main/Tirocinio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INIZIO

Ho incominciato leggendo [l'articolo](https://develweb3.com/en/account-abstraction/) e prendendo nota delle terminologie:

Inizialmemnte l'articolo riporta un breve paragrafo su cosa sia l'**account abstraction** e di come un nuovo tipo di account _smart wallet account_ o semplicemente _smart account_ possa essere usato per estendere la logica dietro i limiti convenzionali di un semplice EOA<sup>[1](#eoa)</sup>.

In che modo l'AA migliorerebbe l'esperienza degli utenti, anche di chi non è familiare con il "mondo delle crypto"?

1. Cambiare il modello di pagamento da uno rigido ad uno flessibile: tipicamente in qualsiasi blockchain il gas viene pagato con una moneta nativa della rete (e.g. ETH). Con l'AA questo vincolo viene rimosso tramite il Paymaster che fa funzionare il tutto come una comune carta di debito e preleva una somma di USDC dal tuo portafoglio per coprire il costo del gas. Il vantaggio è evidente, non si deve più passare da un exchange per comprare piccole frazioni di ETH.

    Per transazioni sponsorizzate si intende che il gas non vien epagato direttamente dall'utente ma da terze parti, e viene fatto questo procedimento:

    UserOperation -> validateUserOp -> validatePaymasterUserOp -> Execution Loop -> postOp.

2. Migliorare il metodo di autenticazione rispetto alla classica _private key_, ma permettere altri metodi quali chiavi di accesso (passkeys), biometrica e multi-firme.

3. Sviluppare **nuovi meccanismi** per il **recupero dei conti** dato che quello attualmente in uso ovvero quello della frase di recupero.

4. Accorpare diverse transazioni un un'unica operazione (e.g. coniatura e trasferimento di un NFT in un'unico ordine).

Ora l'articolo affronta forse l'argomento più **importante** dell'AA:

## ERC-4337 Architecture

ERC-4337 è lo standard più usato per l'AA su Etherium perchè la sua implementazione non necessita alcuna modifica a livello di consenso.

La componentistica principale è formata da:

*   **UserOperations**: sono i precursori delle transazioni convenzionali della blockchain. Contengono informazioni come:
    *   **nonce**: ovvero un contatore sequenziale che contiene il numero di operazioni effettuate. Serve a garantire che ogni operazione venga eseguita una sola volta e nell'ordine corretto.
    *   **valori di pagamento del gas**
    *   dati per interagire con uno _smart contract_
    *   la **firma di autorizzazione delle transazioni**
*   **Bundler**: questi servono per **confezionare la UserOperation** da un mempool, differente rispetto a quello convenzionalmente utilizzato nella blockchain, e mandarlo all'**EntryPoint**

*   **EntryPoint**: uno smart contract _singleton_<sup>[2](#singleton)</sup>  che valida ed esegue le transazioni logiche.

*   **Smart Account**: Uno smart contract che esegue transazioni nella blockchain e viene autorizzato dall'utente ceh controlla l'account. Si comporta come un il portafoglio dell'utente con logica aggiuntiva per migliorare l'esperienza dell'utente.

*   **Paymaster**: uno smart contract _opzionale_ che "sponsorizza" il pagamento del gas delle transazioni per conto degli utenti. Questi accettano anche pagamenti più complessi come in token ERC-20, stablecoin, abbonamenti o pagamenti ricorrenti

*   **Aggregator**: un'entità opzionale per gli smart accounts che cpmvalida e aggrega le firme da più operazioni utente.



### UserOperation

Il punto iniziale dell'interazione utente-blockchain nell'ecosistema AA.

Aspetti fondamentali:

*   Rappresentano una o più azioni da eseguire dallo smart account
*   Contengono dati di verifica delle operazioni
*   Definiscono mome verranno pagate le tasse sul gas
*   Permettono di eseguire più azioni _atomicamente_

### Bundlers

Loro sono responsabili dell'invio di UserOperations all'EntryPoint. Per ogni UserOperation valida, raccolgono una piccola tassa.

Le operazioni utente sono raccolte da uno o più mempool. Idealmente il mempool dovrebbe essere una rete peer-to-peer pubblica e senza permesso.

*   **Rete P2P**: non devono lavorare isolati o comunicare tramite server, devono invece formare una rete dove ogni Bundler parla con gli altri; in pratica se tu invii una UserOperation al "Bundler A", questo la trasmette immediatamente al "Bundler B", "C" e così via. Se il "Bundler A" va offline, la tua operazione è già stata distribuita agli altri nodi della rete che possono processarla.
*   **Pubblica**: la lista d'attesa delle operazioni è pubblica e visibile a tutti. Al giorno d'oggi molti Bundler utilizzano mempool privati, il che rende tutto più veloce ma centralizzato. L'ideale sarebbe quello di garantire maggiore trasparenza e impedire che chiunque possa nascondere male intenzioni.
*   **Senza permesso**: chiunque può decidere di accendere un computer e diventare un Bundler senza dover chiedere autorizzazione ad un'azienda, iniziando a guadagnare impacchettando operazioni. Questo concetto di permissionless evita che si verifichi la **censura** da parte delle aziende che rilasciano queste autorizzazioni.

### EntryPoint

L'EntryPoint agisce come entità centrale nell'architettura ERC-4337. Coordina la verifica e l'esecuzione delle UserOperations. Le due implementazioni in uso sono:

*   **Versione 0.6 (0x5FF137D4FDCD49DCA30c7CF57E578a026d2789)**: La **prima versione** per vedere i casi d'uso reali. I nuovi sviluppi sono incoraggiati a utilizzare l'ultima versione.
*   **Versione 0.7 (0x000000071727De22E5E9d8BAf0edAc6f37da032)**: la versione più recente, introducendo numerosi miglioramenti, anche se è attualmente supportata da pochi provider.



| Caratteristica | EntryPoint v0.6.0 | EntryPoint v0.7.0 | Miglioramento Effettivo |
|:--:|:--:|:--:|:--:|
| Struttura Dati | Unpacked (Piatta) | PackedUserOperation | Riduce i costi di calldata e risolve i limiti di memoria ("stack too deep<sup>[3](#deep)</sup>") della EVM. |
| Gas Paymaster | Condiviso con l'account | Limiti granulari dedicati | Introdotte variabili specifiche per la validazione e la post-esecuzione del paymaster, riducendo i depositi richiesti. |
| Efficienza Gas | Nessuna penale | Penale 10% | Viene addebitato il 10% del gas di esecuzione inutilizzato per scoraggiare stime eccessive che intasano i bundle. |
| Deployment | Campo unico initCode | Split factory/factoryData | Maggiore chiarezza nella creazione dell'account senza necessità di decodifica manuale dei byte. |
| Simulazione | Funzioni on-chain | Spostate off-chain | Alleggerisce il bytecode del contratto EntryPoint, riducendo i costi di interazione. |
| Logica Post-Op | Doppia chiamata possibile | Chiamata singola | Semplifica il flusso di accounting del paymaster e riduce potenziali bug di esecuzione. |
| Supporto L2 |  Solo state overrides | delegateAndRevert() | Permette la simulazione e validazione anche su reti che non supportano modifiche temporanee dello stato. |
| Standardizzazione | Non presente | Supporto ERC-165 | Consente il rilevamento automatico della versione del contratto da parte di dApp e altri smart contract. |

### Smart account
Lo **smart account** funziona come il portafoglio dell’utente, come minimo verifica se una UserOperation sarà accettata durante la fase di verifica.

Siccome stiamo parlando di uno smart account distribuito nella blockchain contiene features aggiuntive come il _pagamento flessibile_ del gas e il _raggruppamento delle transazioni_.

Lo smart account viene rilasciato sulla blockchain **durante la prima transazione**.

#### Differenze tra EOA e smart account

Una differenza chiave sta nella logica di **autenticazione logica della firma** per autorizzare le UserOperation tramite il campo _“firma”_.

Il metodo tradizionale prevede un account supportato dallo standard EIP-1193, il che vuol dire che un **EOA** viene utilizzato per **autorizzare l’esecuzione di UserOperation nello smart account**.

Sono possibili altre alternative più avanzate come le **passkeys** per salvare **chiavi private** in secure enclaves (chip ultra-sicuri isolati dal resto del SO), in modo tale da mantenere una finestra di autorizzazione temporanea.

Gli **smart account** offrono la massima flessibilità e programmabilità nell'ecosistema dell’AA.

#### Problema di standardizzazione

Inizialmente ogni **provider di smart account** sviluppava funzionalità in contratti separati, portando a _problemi di integrazione_ per gli sviluppatori. Facciamo un esempio:
- chiunque creava wallet di un tipo doveva creare codice per il suo recupero, finendo per avere un codice per il recupero diverso per ogni wallet.
- chiunque creava nuove funzionalità per un tipo di wallet, quella funzionalità non funzionava su un wallet creato da un altro provider.

Vedendo questi tipo di problemi, gli sviluppatori di DApps  avrebbero dovuto creare codice diverso per ogni tipo di wallet.

**Soluzione**: i _Moduli_:

- Si è deciso di rendere i wallet vuoti e aggiungere le funzioni tramite moduli.
- I moduli sono come dei **plugin** del wallet che fanno una cosa sola (e.g. un modulo per le session keys, ecc…)

Per far sì che questi moduli siano **interoperabili** è necessario che siano creati tramite **standard** come l'ERC-6900 o l'ERC-7579. Questi standard definiscono:
- Come delegare il comando al modulo
- Quando riceve una transazione, il wallet chiede al modulo se l’utente è abilitato a spendere quell’ammontare di denaro, il modulo risponde e il wallet esegue.

### Gli standard
* **Standard ERC-7579**: descrive l’interfaccia e le funzionalità minime per garantire l’interoperabilità.
* **Standard ERC-6900**: un’implementazione rigorosa che punta a diventare uno **standard** applicando linee specifiche per i **provider**.

#### Quale è meglio?

**Kernel** è un team di di sviluppo di smart account e ha deciso di abbandonare ERC-6900 in favore di ERC-7579, perché?

Sostengono che lo standard ERC-6900 sia troppo pesante e restrittivo, e più che uno standard sembra sia un’implementazione specifica che obbliga tutti a seguire le scelte tecniche di Alchemy.

| Caratteristica | ERC-6900 | ERC-7579 |
|---|---|---|
| Filosofia | Monolitica: È un sistema completo. Se lo usi, devi accettare tutte le decisioni di design di Alchemy. | Minimalista: Si occupa solo di rendere i moduli compatibili. Il resto è libero. |
| Compatibilità | Bassa: Ogni funzione è legata a un solo modulo. Per fare A+B devi scrivere un nuovo modulo AB. | Alta: Puoi combinare più moduli insieme (A e B) per creare logiche complesse senza riscrivere codice. |
| Session Keys | Devono essere attivate on-chain (pagando gas subito). | Possono essere attivate off-chain (firmi un messaggio e paghi gas solo quando le usi). |
| Flessibilità | Vieta alcune funzioni avanzate di ERC-4337 (come gli Aggregators). | Supporta pienamente tutte le funzioni di ERC-4337. |
| Origine | Creato da un solo team (Alchemy) quando c'era poca esperienza pratica. | Creato da chi gestisce milioni di account reali, basandosi sull'esperienza d'uso. |

### Paymaster

I **Paymasters** sono **_smart contracts_** che offrono maggiore flessibilità nelle **opzioni di pagamento del gas**. Possibili casi d’uso sono:
- **Transazioni sponsorizzate** sotto specifiche regole di servizio, permette agli utenti di pagare senza gas.
- Accettano **ERC-20 tokens** per pagare le tasse sul gas
- Sviluppano **modelli di pagamento** basati su abbonamento.

Gli **ERC-20 tokens** sono creati tramite **smart contract**, a differenza di ETH che è la moneta nativa della blockchain, ERC, che sta per Ethereum Request for Comments, sono token **standardizzati** che equivalgono a Stablecoins come USDC ecc.

I **tokens** non risiedono all’interno del nostro wallet, all’interno, il wallet contiene solo la chiave privata, i tokens sono invece annotazioni in un **registro (ledger)** tenuto dallo smart contract sulla blockchain.

La particolarità dei tokens è che possono essere inviati da un indirizzo ad un altro.

#### Altri standard correlati:

* **EIP-7702**: Una proposta per **convertire temporaneamente un EOA** in uno **smart account**, consentendo agli utenti di _rappresentare il proprio EOA con la logica del contratto intelligente_.
* **EIP-5792** : Mira a **unificare** il modo in cui le **DApp** interagiscono con diversi tipi di portafogli (EOA o account intelligenti) per **mitigare i problemi di frammentazione** che ostacolano l'adozione del **web3**. introduce un'API standardizzata.

### Note:
* <a id="eoa"></a> Nota **1**: EOA = External Owned Account.
* <a id="singleton"></a> Nota **2**: singleton = è uno smart contract **univoco** per tutta la blockchain.
* <a id="deep"></a> Nota **3**: stack too deep = limitazione tecnica imposta dalla Ethereum Virtual Machine (EVM).

# Simple-AA-Wallet

Dal seguente link di [GitHub](https://github.com/0xEunum/Simple-AA-Wallet) inizio leggendo il readme per capire di cosa si tratta, come è strutturato e come fare un’implementazione locale:
Sin dall’inizio si notano delle caratteristiche importanti:

* Uno **Smart Account (SA)** viene deployato con un `EntryPoint`
* Il **deployer diventa il proprietario** dello smart account
* Solo il **proprietario del EOA** può firmare `UserOperation`s
* Le firme sono valide nel `validateUserOp`
* Operazioni valide vengono eseguite via `execute(...)`
* La proprietà (Ownership) può essere trasferita ad un altro EOA

Ci sono **3 tipi di contratto**:

1. EntryPoint
    * Standard ERC-4337 EntryPoint
    * Deployed locally
    * Used by both Smart Account and Bundler
2. Smart Account
    * Stores `owner`
    * Validates signatures in `validateUserOp`
    * Executes calls via `execute(address target, uint256 value, bytes calldata data)`
    * Supports ownership transfer
3. Counter (Target Contract)
    * Simple contract with `increment()`
    * Used to prove execution and state change


Inizio con il setup locale:
Seguendo il readme per il setup mi trovo davanti ad un problema:

```bash
~/Simple-AA-Wallet$ cd infra/bundler/
~/Simple-AA-Wallet/infra/bundler$ yarn && yarn preprocess
00h00m00s 0/0: : ERROR: There are no scenarios; must have at least one.
```

ho riscontrato parecchi problemi con questo comando non mi chiarisco perchè, dopo diversi tentativi ho risolto e sono riuscito ad eseguire il comando

In [ ]:
anvil --disable-code-size-limit --chain-id 8546

## Risoluzione problema con yarn

Non so se sia necessario ma scrivo la risoluzione di questo problema:

NB: il mio SO è Ubuntu

```bash
/Simple-AA-Wallet/infra/bundler$ yarn && yarn preprocess
Comando «yarn» non trovato, ma può essere installato con:
sudo apt install cmdtest
```

allora inserisco il comando `sudo apt install cmdtest`

ma il comando `yarn && yarn preprocess` ancora non funziona allora digito

```bash
/Simple-AA-Wallet$ npm install
up to date, audited 47 packages in 749ms
15 packages are looking for funding
  run `npm fund` for details
found 0 vulnerabilities
```

il problema sembra essere che mancano proprio dei pacchetti, rimedio scrivendo:

In [ ]:
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - sudo apt-get install -y nodejs

Cancello eventuali tracce sporche:

In [ ]:
rm -rf node_modules package-lock.json yarn.lock

Installo NPM senza conflitti di nome

In [ ]:
npm install

e infine

In [ ]:
npm run preprocess

aggirando il problema del comando yarn

L’output però risulta avere TypeError
``` bash
TypeError: The "mcopy" instruction is only available for Cancun-compatible VMs (you are currently compiling for "paris").
```

Nella seguente directory eseguo

```bash
/Simple-AA-Wallet/infra/bundler/packages/bundler$ nano hardhat.config.ts
```

e nella sezione dedicata a solidity del file inserisco dopo optimizer questa riga
`evmVersion: "cancun"`
Compilo


In [ ]:
npx hardhat compile

e torno alla cartella `/Simple-AA-Wallet/`

Non avendo installato anvil procedo con l’installazione:

In [ ]:
curl -L https://foundry.paradigm.xyz | bash

poi

In [ ]:
source ~/.bashrc

Infine installo `foundryup`

In [ ]:
foundryup

e avvio anvil come da README

In [ ]:
anvil --disable-code-size-limit --chain-id 8546

L’output del comando è
```bash
                             _   _
                            (_) | |
      __ _   _ __   __   __  _  | |
     / _` | | '_ \  \ \ / / | | | |
    | (_| | | | | |  \ V /  | | | |
     \__,_| |_| |_|   \_/   |_| |_|

    1.5.1-stable (b0a9dd9ced 2025-12-22T11:39:01.425730780Z)
    https://github.com/foundry-rs/foundry

Available Accounts
==================

(0) 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266 (10000.000000000000000000 ETH)
(1) 0x70997970C51812dc3A010C7d01b50e0d17dc79C8 (10000.000000000000000000 ETH)
(2) 0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC (10000.000000000000000000 ETH)
(3) 0x90F79bf6EB2c4f870365E785982E1f101E93b906 (10000.000000000000000000 ETH)
(4) 0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65 (10000.000000000000000000 ETH)
(5) 0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc (10000.000000000000000000 ETH)
(6) 0x976EA74026E726554dB657fA54763abd0C3a0aa9 (10000.000000000000000000 ETH)
(7) 0x14dC79964da2C08b23698B3D3cc7Ca32193d9955 (10000.000000000000000000 ETH)
(8) 0x23618e81E3f5cdF7f54C3d65f7FBc0aBf5B21E8f (10000.000000000000000000 ETH)
(9) 0xa0Ee7A142d267C1f36714E4a8F75612F20a79720 (10000.000000000000000000 ETH)

Private Keys
==================

(0) 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80
(1) 0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d
(2) 0x5de4111afa1a4b94908f83103eb1f1706367c2e68ca870fc3fb9a804cdab365a
(3) 0x7c852118294e51e653712a81e05800f419141751be58f605c371e15141b007a6
(4) 0x47e179ec197488593b187f80a00eb0da91f1b9d0b13f8733639f19c30a34926a
(5) 0x8b3a350cf5c34c9194ca85829a2df0ec3153be0318b5e2d3348e872092edffba
(6) 0x92db14e403b83dfe3df233f83dfa3a0d7096f21ca9b0d6d6b8d88b2b4ec1564e
(7) 0x4bbbf85ce3377467afe5d46f804f221813b2bb87f24d81f60f1fcdbf7cbf4356
(8) 0xdbda1821b80551c9d65939329250298aa3472ba22feea921c0cf5d620ea67b97
(9) 0x2a871d0798f97d79848a013d4936a73bf4cc922c825d33c1cf7073dff6d409c6
Wallet
==================
Mnemonic:          test test test test test test test test test test test junk
Derivation path:   m/44'/60'/0'/0/


Chain ID
==================

8546

Base Fee
==================

1000000000

Gas Limit
==================

30000000

Genesis Timestamp
==================

1773149770

Genesis Number
==================

0

Listening on 127.0.0.1:8545
```

## Tentativo 1

### Deployment delle varie entità della blockchain

Apro un nuovo terminale lasciando aperto quello dove ho avviato la blockchain in locale e proseguo:

#### EntryPoint

Il README dice:

```bash
forge script script/Deploy.s.sol:DeployEntryPoint \
  --private-key <PRIVATE_KEY> \
  --rpc-url anvil \
  --broadcast
```

Siccome il README dice chiaramente _“Use Anvil Account-1 for all deployments.”_, tramite l’output riportato prima troviamo che all’Account 1 è associata la PRIVATE_KEY `0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d`.

Al posto di anvil uso l’indirizzo che mi ha restituito: `127.0.0.1:8545` (ovvero l’indirizzo locale)

Quindi digito:

In [ ]:
forge script script/Deploy.s.sol:DeployEntryPoint \
--private-key 0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d \
--rpc-url http://127.0.0.1:8545 \
--broadcast


Il terminale restituisce `ONCHAIN EXECUTION COMPLETE & SUCCESSFUL`.

E il suo indirizzo:

Contract Address: `0x8464135c8F25Da09e49BC8782676a84730C318bC`

#### Smart Account

Il README prosegue:
```bash
forge create src/SmartAccount/MinimalAccount.sol:MinimalAccount \
  --private-key <PRIVATE_KEY> \
  --rpc-url anvil \
  --constructor-args <ENTRY_POINT_ADDRESS> \
  --broadcast
```
In `<PRIVATE_KEY>` dovremo inserire l’indirizzo di Account-1
Al posto di `anvil` sempre il nostro indirizzo in locale
E come `<ENTRY_POINT_ADDRESS>` l’indirizzo dell’EntryPoint che abbiamo avuto prima
Quindi:

In [ ]:
forge create src/SmartAccount/MinimalAccount.sol:MinimalAccount --broadcast --private-key 0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d --rpc-url http://127.0.0.1:8545 --constructor-args 0x8464135c8F25Da09e49BC8782676a84730C318bC

NB: in questo comando ho dovuto inserire `--broadcast` in mezzo perché forge è capriccioso

Ottenendo come output

```bash
Deployer: 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
Deployed to: 0x71C95911E9a5D330f4D621842EC243EE1343292e
Transaction hash: 0xbd52b2b5ed2baadcb77cdbb03cdde9411ef974b7ebb329e7b02099ca074ec9b9
```

#### Counter contract

Il README dice:
```bash
forge script script/Deploy.s.sol:DeployCounter \
  --private-key <PRIVATE_KEY> \
  --rpc-url anvil \
  --broadcast
```

E compilo i campi vuoti come ho già fatto:


In [ ]:
forge script script/Deploy.s.sol:DeployCounter \
  --private-key 0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d \
  --rpc-url http://127.0.0.1:8545 \
  --broadcast

Rilasciando come output:
```bash
[⠊] Compiling...
No files changed, compilation skipped
Script ran successfully.

== Return ==
0: contract Counter 0x948B3c65b89DF0B4894ABE91E6D02FE579834F8F

## Setting up 1 EVM.

==========================

Chain 8546

Estimated gas price: 1.865714885 gwei

Estimated total gas used for script: 204699

Estimated amount required: 0.000381909971244615 ETH

==========================

##### 8546
✅  [Success] Hash: 0x8a2aa39192a671a9b9adfc8a64fdc73126fa0ccdc942a912d3d63108045e4c58
Contract Address: 0x948B3c65b89DF0B4894ABE91E6D02FE579834F8F
Block: 3
Paid: 0.000129646940474277 ETH (157461 gas * 0.823359057 gwei)

✅ Sequence #1 on 8546 | Total Paid: 0.000129646940474277 ETH (157461 gas * avg 0.823359057 gwei)

==========================

ONCHAIN EXECUTION COMPLETE & SUCCESSFUL.

Transactions saved to: /home/mikitro/Desktop/Simple-AA-Wallet/broadcast/Deploy.s.sol/8546/run-latest.json

Sensitive values saved to: /home/mikitro/Desktop/Simple-AA-Wallet/cache/Deploy.s.sol/8546/run-latest.json
```

### Riepilogo
Abbiamo 4 indirizzi:
* Account-1 (Private key):
  0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d
* EntryPoint:
  0x8464135c8F25Da09e49BC8782676a84730C318bC
* Smart Account:
  0x71C95911E9a5D330f4D621842EC243EE1343292e
* Counter:
  0x948B3c65b89DF0B4894ABE91E6D02FE579834F8F

## Punto 5: Verifica degli indirizzi

In `/Simple-AA-Wallet/src$` apro il file `index.ts` e aggiorno le costanti degli indirizzi in:

```bash
const ENTRY_POINT = "0x8464135c8F25Da09e49BC8782676a84730C318bC";
const ACCOUNT = "0x71C95911E9a5D330f4D621842EC243EE1343292e";
const COUNTER = "0x948B3c65b89DF0B4894ABE91E6D02FE579834F8F";
```

Dentro `/Simple-AA-Wallet/userop/runners$` trovo tre file da modificare:
```bash
runBuildUserOp.ts  runSendUserOp.ts  runSignUserOp.ts
```
Sempre con gli indirizzi sopra riportati. A cosa servono questi file?



| File | Cosa fa | Quando usarlo |
|---|---|---|
| `runBuildUserOp.ts` | Crea la struttura dati della transazione (UserOp) ma non la firma. | Per verificare che il gas e i dati della chiamata siano calcolati bene. |
| `runSignUserOp.ts` | Crea la UserOp e aggiunge la tua firma crittografica. | Per testare se la tua chiave privata produce una firma valida. |
| `runSendUserOp.ts` | Crea, firma e invia tutto al Bundler. | È il test completo per vedere l'operazione che arriva sulla blockchain. |

Punto 6: Trovare lo Smart Account
Il README dice:
```bash
cast send <SMART_ACCOUNT_ADDRESS> \
  --value 1ether \
  --private-key <PRIVATE_KEY> \
  --rpc-url anvil
```
Quindi:

In [ ]:
cast send 0x71C95911E9a5D330f4D621842EC243EE1343292e \
--value 1ether \
--private-key 0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d \
--rpc-url http://127.0.0.1:8545


Ottenendo come output:
```bash
blockHash            0x15e42f7bcaa8c99fc14084bf34d74c6a69f1f4ac340093ebd66c23e7398aed86
blockNumber          4
contractAddress      
cumulativeGasUsed    21055
effectiveGasPrice    721519567
from                 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
gasUsed              21055
logs                 []
logsBloom            0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
root                 
status               1 (success)
transactionHash      0x24595935d7cbd9df31b22f42df647bad1a811fe4bfa5f319d2e6661c879b8ebd
transactionIndex     0
type                 2
blobGasPrice         1
blobGasUsed          
to                   0x71C95911E9a5D330f4D621842EC243EE1343292e
```

Per controllare il saldo dello Smart Account inseriamo il comando:

In [ ]:
cast balance 0x71C95911E9a5D330f4D621842EC243EE1343292e --rpc-url http://127.0.0.1:8545

E dovremmo visualizzare l’importo in wei pari a: `1000000000000000000`
Il secondo comando che troviamo nel punto 6 corrisponde ad un modo più sicuro per inviare transazioni rispetto ad inserire la Private Key in chiaro. Per completezza la riporto:
```bash
cast send <SMART_ACCOUNT_ADDRESS> \
  --value 1ether \
  --account <ACCOUNT_NAME> \
  --rpc-url anvil
```

### Punto 7: Variabili di ambiente

Ritorno alla root: `/Simple-AA-Wallet/`

E seguendo quello che dice il README creo un file nascosto chiamato `.env`, e ci scrivo:

```txt
OWNER_PRIVATE_KEY=<ANVIL_ACCOUNT_1_PRIVATE_KEY>
```

### Punto 8: Start the local bundler

Dentro la cartella `/Simple-AA-Wallet/infra/` bundler inserisco il seguente comando:
```bash
yarn run bundler \
  --unsafe \
  --auto \
  --entryPoint <ENTRY_POINT_ADDRESS>
```
Inserendo come `<ENTRY_POINT_ADDRESS>` quello dell’EntryPoint, dunque:

In [ ]:
yarn run bundler \
--unsafe \
--auto \
--entryPoint 0x8464135c8F25Da09e49BC8782676a84730C318bC

Avendo come output (di errore) questo

```bash
~/Desktop/Simple-AA-Wallet/infra/bundler$ yarn run bundler \
  --unsafe \
  --auto \
  --entryPoint 0x8464135c8F25Da09e49BC8782676a84730C318bC
yarn run v1.22.22
$ yarn --cwd packages/bundler bundler --unsafe --auto --entryPoint 0x8464135c8F25Da09e49BC8782676a84730C318bC
$ TS_NODE_TRANSPILE_ONLY=1 ts-node ./src/exec.ts --config ./localconfig/bundler.config.json --unsafe --auto --entryPoint 0x8464135c8F25Da09e49BC8782676a84730C318bC
{ '1': { 'EREP-010': { enabled: true, exceptions: [Array] } } }
command-line arguments:  {
  config: './localconfig/bundler.config.json',
  auto: true,
  unsafe: true,
  entryPoint: '0x8464135c8F25Da09e49BC8782676a84730C318bC'
}
Aborted: Cannot read properties of undefined (reading 'existsSync')
error Command failed with exit code 1.
info Visit https://yarnpkg.com/en/docs/cli/run for documentation about this command.
error Command failed with exit code 1.
info Visit https://yarnpkg.com/en/docs/cli/run for documentation about this command.
```

HO RISOLTO IL PROBLEMA!!

Allora:

Ho ipotizzato che il problema risiedesse in `./localconfig/bundler.config.json`
e siccome spesso può capitare che anche se passo l’entrypoint via riga di comando, alcuni bundler leggono comunque il config, allora ho modificato il config inserendo il mio indirizzo

Non ho risolto così, mi ha dato un altro problema

```bash
~/Desktop/Simple-AA-Wallet/infra/bundler$ yarn run bundler   --unsafe   --auto   --entryPoint 0x8464135c8F25Da09e49BC8782676a84730C318bC
yarn run v1.22.22
$ yarn --cwd packages/bundler bundler --unsafe --auto --entryPoint 0x8464135c8F25Da09e49BC8782676a84730C318bC
$ TS_NODE_TRANSPILE_ONLY=1 ts-node ./src/exec.ts --config ./localconfig/bundler.config.json --unsafe --auto --entryPoint 0x8464135c8F25Da09e49BC8782676a84730C318bC
{ '1': { 'EREP-010': { enabled: true, exceptions: [Array] } } }
command-line arguments:  {
  config: './localconfig/bundler.config.json',
  auto: true,
  unsafe: true,
  entryPoint: '0x8464135c8F25Da09e49BC8782676a84730C318bC'
}
Aborted: Cannot read properties of undefined (reading 'existsSync')
error Command failed with exit code 1.
info Visit https://yarnpkg.com/en/docs/cli/run for documentation about this command.
error Command failed with exit code 1.
info Visit https://yarnpkg.com/en/docs/cli/run for documentation about this command.
```

L’errore che ho ottenuto ora è Cannot read properties of undefined (reading 'existsSync')
Per risolvere questo problema scrivo:

```bash
~/Desktop/Simple-AA-Wallet/infra/bundler$ npx @pimlico/alto \
  --rpc-url http://127.0.0.1:8545 \
  --entrypoints 0x8464135c8F25Da09e49BC8782676a84730C318bC \
  --executor-private-keys 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80 \
  --utility-private-key 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80 \
  --unsafe
Rilasciando un output lunghissimo con un errore: porta 3000 già in uso
Termino il processo che occupa la porta 3000 con: fuser -k 3000/tcp
E rieseguo il comando
npx @pimlico/alto \
  --rpc-url http://127.0.0.1:8545 \
  --entrypoints 0x8464135c8F25Da09e49BC8782676a84730C318bC \
  --executor-private-keys 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80 \
  --utility-private-key 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80 \
  --unsafe
```

PROBLEMA RISOLTO
L’output ricevuto è lunghissimo, contiene infiniti numeri esadecimali alternati da pezzi di codice.
Verso la fine l’output risulta:

```bash
[17:35:12.393] INFO (44590): Created memory sender manager
    module: "sender-manager"
[17:35:12.395] INFO (44590): received response
    module: "public_client"
    body: {
      "method": "eth_getBalance",
      "params": [
        "0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266",
        "latest"
      ]
    }
    result: "0x21e0b9f2fc9ba41146d"
[17:35:12.397] INFO (44590): Using memory for outstanding mempool
    module: "mempool-store"
[17:35:12.397] INFO (44590): Using memory for user operation receipt cache
    module: "receipt-cache"
[17:35:12.398] INFO (44590): Initialized 1 executor wallets
    module: "root"
[17:35:12.415] INFO (44590): Server listening at http://0.0.0.0:3000
    module: "rpc"
[17:35:27.399] INFO (44590): received response
    module: "public_client"
    body: {
      "method": "eth_getBalance",
      "params": [
        "0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266",
        "latest"
      ]
    }
    result: "0x21e0b9f2fc9ba41146d"
[17:35:42.400] INFO (44590): received response
    module: "public_client"
    body: {
      "method": "eth_getBalance",
      "params": [
        "0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266",
        "latest"
      ]
    }
    result: "0x21e0b9f2fc9ba41146d"
```

Alla fine ho scoperto di non aver risolto
testando vari comandi per trasferire fondi noto un errore

`HTTP request failed.`

Torno al punto 3 e riavvio anvil

# Tentativo 2

## Avvio di Anvil

```bash
~/Desktop/Simple-AA-Wallet$ anvil \
  --disable-code-size-limit \
  --chain-id 8546

                             _   _
                            (_) | |
      __ _   _ __   __   __  _  | |
     / _` | | '_ \  \ \ / / | | | |
    | (_| | | | | |  \ V /  | | | |
     \__,_| |_| |_|   \_/   |_| |_|

    1.5.1-stable (b0a9dd9ced 2025-12-22T11:39:01.425730780Z)
    https://github.com/foundry-rs/foundry

Available Accounts
==================

(0) 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266 (10000.000000000000000000 ETH)
(1) 0x70997970C51812dc3A010C7d01b50e0d17dc79C8 (10000.000000000000000000 ETH)
(2) 0x3C44CdDdB6a900fa2b585dd299e03d12FA4293BC (10000.000000000000000000 ETH)
(3) 0x90F79bf6EB2c4f870365E785982E1f101E93b906 (10000.000000000000000000 ETH)
(4) 0x15d34AAf54267DB7D7c367839AAf71A00a2C6A65 (10000.000000000000000000 ETH)
(5) 0x9965507D1a55bcC2695C58ba16FB37d819B0A4dc (10000.000000000000000000 ETH)
(6) 0x976EA74026E726554dB657fA54763abd0C3a0aa9 (10000.000000000000000000 ETH)
(7) 0x14dC79964da2C08b23698B3D3cc7Ca32193d9955 (10000.000000000000000000 ETH)
(8) 0x23618e81E3f5cdF7f54C3d65f7FBc0aBf5B21E8f (10000.000000000000000000 ETH)
(9) 0xa0Ee7A142d267C1f36714E4a8F75612F20a79720 (10000.000000000000000000 ETH)

Private Keys
==================

(0) 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80
(1) 0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d
(2) 0x5de4111afa1a4b94908f83103eb1f1706367c2e68ca870fc3fb9a804cdab365a
(3) 0x7c852118294e51e653712a81e05800f419141751be58f605c371e15141b007a6
(4) 0x47e179ec197488593b187f80a00eb0da91f1b9d0b13f8733639f19c30a34926a
(5) 0x8b3a350cf5c34c9194ca85829a2df0ec3153be0318b5e2d3348e872092edffba
(6) 0x92db14e403b83dfe3df233f83dfa3a0d7096f21ca9b0d6d6b8d88b2b4ec1564e
(7) 0x4bbbf85ce3377467afe5d46f804f221813b2bb87f24d81f60f1fcdbf7cbf4356
(8) 0xdbda1821b80551c9d65939329250298aa3472ba22feea921c0cf5d620ea67b97
(9) 0x2a871d0798f97d79848a013d4936a73bf4cc922c825d33c1cf7073dff6d409c6

Wallet
==================
Mnemonic:          test test test test test test test test test test test junk
Derivation path:   m/44'/60'/0'/0/


Chain ID
==================

8546

Base Fee
==================

1000000000

Gas Limit
==================

30000000

Genesis Timestamp
==================

1773220000

Genesis Number
==================

0

Listening on 127.0.0.1:8545
```




Notiamo l’account 1 con indirizzo:

`0x70997970C51812dc3A010C7d01b50e0d17dc79C8`

e Private Key:

`0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d`

## 4.1 Deploy EntryPoint
```bash
forge script script/Deploy.s.sol:DeployEntryPoint \
  --private-key <PRIVATE_KEY> \
  --rpc-url anvil \
  --broadcast
```

Come ieri sistemo i campi vuoti
```bash
forge script script/Deploy.s.sol:DeployEntryPoint \
  --private-key 0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d \
  --rpc-url 127.0.0.1:8545 \
  --broadcast
```

Restituendo come output:

```bash
[⠊] Compiling...
No files changed, compilation skipped
Script ran successfully.

== Return ==
0: contract EntryPoint 0x8464135c8F25Da09e49BC8782676a84730C318bC
Error: `EntryPoint` is above the contract size limit (29425 > 24576).
Do you wish to continue? yes

## Setting up 1 EVM.

==========================

Chain 8546

Estimated gas price: 2.000000001 gwei

Estimated total gas used for script: 9025760

Estimated amount required: 0.01805152000902576 ETH

==========================

##### 8546
✅  [Success] Hash: 0x3f9a0e2bf105ede6111078bca6a9d691b1579b120a87e47599221722ed4ec010
Contract Address: 0x8464135c8F25Da09e49BC8782676a84730C318bC
Block: 1
Paid: 0.006942893006942893 ETH (6942893 gas * 1.000000001 gwei)

✅ Sequence #1 on 8546 | Total Paid: 0.006942893006942893 ETH (6942893 gas * avg 1.000000001 gwei)
                                                                                

==========================

ONCHAIN EXECUTION COMPLETE & SUCCESSFUL.

Transactions saved to: /home/mikitro/Desktop/Simple-AA-Wallet/broadcast/Deploy.s.sol/8546/run-latest.json

Sensitive values saved to: /home/mikitro/Desktop/Simple-AA-Wallet/cache/Deploy.s.sol/8546/run-latest.json
```

## 4.2 Deploy Smart Account

Il readme dice di inserire
```bash
forge create src/SmartAccount/MinimalAccount.sol:MinimalAccount \
  --private-key <PRIVATE_KEY> \
  --rpc-url anvil \
  --constructor-args <ENTRY_POINT_ADDRESS> \
  --broadcast
```

Sistemandolo esce:

In [ ]:
forge create src/SmartAccount/MinimalAccount.sol:MinimalAccount  --broadcast   --private-key 0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d   --rpc-url 127.0.0.1:8545   --constructor-args 0x8464135c8F25Da09e49BC8782676a84730C318bC

NB il comando darà sempre problemi, il motivo non mi è chiaro ma il problema risiede nell’opzione --broadcast, dice di aggiungerlo quando c’è già, allora l’ho messo come prima opzione e funziona

avendo come output:

```bash
[⠊] Compiling...
No files changed, compilation skipped
Deployer: 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
Deployed to: 0x71C95911E9a5D330f4D621842EC243EE1343292e
Transaction hash: 0xbd52b2b5ed2baadcb77cdbb03cdde9411ef974b7ebb329e7b02099ca074ec9b9
```

Notiamo che l’indirizzo dello SmartAccount è:

`0x71C95911E9a5D330f4D621842EC243EE1343292e`

E il deployer è Account-1


## 4.3 Deploy Counter contract

Il readme dice:
```bash
forge script script/Deploy.s.sol:DeployCounter \
  --private-key <PRIVATE_KEY> \
  --rpc-url anvil \
  --broadcast
```
Sistemato esce:

In [ ]:
forge script script/Deploy.s.sol:DeployCounter \
  --private-key 0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d \
  --rpc-url 127.0.0.1:8545 \
  --broadcast

ottenendo come output:

```bash
[⠊] Compiling...
No files changed, compilation skipped
Script ran successfully.

== Return ==
0: contract Counter 0x948B3c65b89DF0B4894ABE91E6D02FE579834F8F

## Setting up 1 EVM.

==========================

Chain 8546

Estimated gas price: 1.865714885 gwei

Estimated total gas used for script: 204699

Estimated amount required: 0.000381909971244615 ETH

==========================

##### 8546
✅  [Success] Hash: 0x8a2aa39192a671a9b9adfc8a64fdc73126fa0ccdc942a912d3d63108045e4c58
Contract Address: 0x948B3c65b89DF0B4894ABE91E6D02FE579834F8F
Block: 3
Paid: 0.000129646940474277 ETH (157461 gas * 0.823359057 gwei)

✅ Sequence #1 on 8546 | Total Paid: 0.000129646940474277 ETH (157461 gas * avg 0.823359057 gwei)
                                                                                

==========================

ONCHAIN EXECUTION COMPLETE & SUCCESSFUL.

Transactions saved to: /home/mikitro/Desktop/Simple-AA-Wallet/broadcast/Deploy.s.sol/8546/run-latest.json

Sensitive values saved to: /home/mikitro/Desktop/Simple-AA-Wallet/cache/Deploy.s.sol/8546/run-latest.json
```

## Punto 5: Verify deployed addresses

Ora devo modificare le costanti del file `src/index.ts` e dei file dentro `userop/runners`

Ok fatto

Una miglioria che si può apportare al progetto è l’aggiunta automatica di questi indirizzi all’interno di questi 4 file totali

## Punto 6: Fund the Smart Account

Il readme dice:
```bash
cast send <SMART_ACCOUNT_ADDRESS> \
  --value 1ether \
  --private-key <PRIVATE_KEY> \
  --rpc-url anvil
```
Sistemato


In [ ]:
cast send 0x71C95911E9a5D330f4D621842EC243EE1343292e \
  --value 1ether \
  --private-key 0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d \
  --rpc-url 127.0.0.1:8545

Ottenendo come output:

```bash
blockHash            0xe3bfedb3ef44a5c4ad19707ae7fbca14a14c3df43e14563ef5772b7c55fcddca
blockNumber          4
contractAddress      
cumulativeGasUsed    21055
effectiveGasPrice    721519567
from                 0x70997970C51812dc3A010C7d01b50e0d17dc79C8
gasUsed              21055
logs                 []
logsBloom            0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
root                 
status               1 (success)
transactionHash      0x24595935d7cbd9df31b22f42df647bad1a811fe4bfa5f319d2e6661c879b8ebd
transactionIndex     0
type                 2
blobGasPrice         1
blobGasUsed          
to                   0x71C95911E9a5D330f4D621842EC243EE1343292e
```

## Punto 7: Environment variables
creo il file nascosto .env

ci scrivo

```txt
OWNER_PRIVATE_KEY=0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d
```

## Punto 8: Start the local bundler
Il readme dice

In [ ]:
cd infra/bundler

e poi:

```bash
yarn run bundler \
  --unsafe \
  --auto \
  --entryPoint <ENTRY_POINT_ADDRESS>
```

sistemato:

In [ ]:
yarn run bundler \
  --unsafe \
  --auto \
  --entryPoint 0x8464135c8F25Da09e49BC8782676a84730C318bC

Ottengo questo output:

```bash
yarn run v1.22.22
$ yarn --cwd packages/bundler bundler --unsafe --auto --entryPoint 0x8464135c8F25Da09e49BC8782676a84730C318bC
$ TS_NODE_TRANSPILE_ONLY=1 ts-node ./src/exec.ts --config ./localconfig/bundler.config.json --unsafe --auto --entryPoint 0x8464135c8F25Da09e49BC8782676a84730C318bC
{ '1': { 'EREP-010': { enabled: true, exceptions: [Array] } } }
command-line arguments:  {
  config: './localconfig/bundler.config.json',
  auto: true,
  unsafe: true,
  entryPoint: '0x8464135c8F25Da09e49BC8782676a84730C318bC'
}
Aborted: Cannot read properties of undefined (reading 'existsSync')
error Command failed with exit code 1.
info Visit https://yarnpkg.com/en/docs/cli/run for documentation about this command.
error Command failed with exit code 1.
info Visit https://yarnpkg.com/en/docs/cli/run for documentation about this command.
```

Ok, ci è voluto parecchio tempo

Il problema risiede nel fatto che lo stesso progetto utilizza due standard differenti: lo 0.6 e lo 0.7. Ho dovuto modificare il file `src/index.ts` che pubblicherò in una mia repo di github assieme a tutto il progetto sistemato

## Punto 9: Run the UserOperation flow

Ora bisogna eseguire il comando npm run start

Solo così non funziona bisogna bypassare problemi di alto inserendo questo comando

```bash
~/Desktop/Simple-AA-Wallet/infra/bundler/packages/bundler/localconfig$ npx @pimlico/alto   --rpc-url http://127.0.0.1:8545   --entrypoints 0x8464135c8F25Da09e49BC8782676a84730C318bC   --executor-private-keys 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80   --utility-private-key 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80   --port 3000   --skip-simulation   --unsafe
```

poi in un altro terminale npm run start che, con il file src/index.ts sistemato dovrebbe tornare:

```bash
~/Desktop/Simple-AA-Wallet$ npm run start

> simple-aa-wallet@1.0.0 start
> ts-node src/index.ts

--- 1. Building UserOp ---
--- 2. Signing ---
--- 3. Executing handleOps Directly ---
🚀 SUCCESS! Transaction sent.
Transaction Hash: 0xf7b18929c258de295f1aba1a9b10aeb24b95a44408eb729a956da3528fb9aceb
✅ CONFIRMED in block: 14
```

Il readme prosegue chiedendo di inserire .count(), ma non funziona, mentre getNumber() sì:

In [ ]:
cast call 0x948B3c65b89DF0B4894ABE91E6D02FE579834F8F "getNumber()" --rpc-url http://127.0.0.1:8545

`0x0000000000000000000000000000000000000000000000000000000000000001`

Ora ci sono comandi da inserire per debugging e learning, partendo con

In [ ]:
npm run build:userop

Ottenendo:
``` bash
> simple-aa-wallet@1.0.0 build:userop
> ts-node userop/runners/runBuildUserOp.ts

UserOp build: {
  sender: '0x70997970C51812dc3A010C7d01b50e0d17dc79C8',
  nonce: 0n,
  initCode: '0x',
  callData: '0xb61d27f6000000000000000000000000948b3c65b89df0b4894abe91e6d02fe579834f8f000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000600000000000000000000000000000000000000000000000000000000000000004d09de08a00000000000000000000000000000000000000000000000000000000',
  accountGasLimits: '0x00000000000000000000000000000000000000000000000000000000000493e0',
  preVerificationGas: 50000n,
  gasFees: '0x00000000000000000000000000000000000000000000000000000000004c4b40',
  paymasterAndData: '0x',
  signature: '0x'
}
```

poi di inserire

In [ ]:
npm run sign:userop

restituendo:

```bash
> simple-aa-wallet@1.0.0 sign:userop
> ts-node userop/runners/runSignUserOp.ts

PackedUserOp build {
  sender: '0x70997970C51812dc3A010C7d01b50e0d17dc79C8',
  nonce: 0n,
  initCode: '0x',
  callData: '0xb61d27f6000000000000000000000000948b3c65b89df0b4894abe91e6d02fe579834f8f000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000600000000000000000000000000000000000000000000000000000000000000004d09de08a00000000000000000000000000000000000000000000000000000000',
  accountGasLimits: '0x00000000000000000000000000000000000000000000000000000000000493e0',
  preVerificationGas: 50000n,
  gasFees: '0x00000000000000000000000000000000000000000000000000000000004c4b40',
  paymasterAndData: '0x',
  signature: '0x'
}
UserOp signed: {
  sender: '0x70997970C51812dc3A010C7d01b50e0d17dc79C8',
  nonce: 0n,
  initCode: '0x',
  callData: '0xb61d27f6000000000000000000000000948b3c65b89df0b4894abe91e6d02fe579834f8f000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000600000000000000000000000000000000000000000000000000000000000000004d09de08a00000000000000000000000000000000000000000000000000000000',
  accountGasLimits: '0x00000000000000000000000000000000000000000000000000000000000493e0',
  preVerificationGas: 50000n,
  gasFees: '0x00000000000000000000000000000000000000000000000000000000004c4b40',
  paymasterAndData: '0x',
  signature: '0x403e079bec12e08d822af12c0b7e4c9208b81efe39a5e247fe371e1b33e0dc5601e6e658f7f01cc7253448ed28bc315a6b1cf3f4aeb562778ac5dd91841b1b231c'
}
```

Infine

In [ ]:
npm run send:userop

Ottenendo:

```bash
> simple-aa-wallet@1.0.0 send:userop
> ts-node userop/runners/runSendUserOp.ts

PackedUserOp build: {
  sender: '0x70997970C51812dc3A010C7d01b50e0d17dc79C8',
  nonce: 0n,
  initCode: '0x',
  callData: '0xb61d27f6000000000000000000000000948b3c65b89df0b4894abe91e6d02fe579834f8f000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000600000000000000000000000000000000000000000000000000000000000000004d09de08a00000000000000000000000000000000000000000000000000000000',
  accountGasLimits: '0x00000000000000000000000000000000000000000000000000000000000493e0',
  preVerificationGas: 50000n,
  gasFees: '0x00000000000000000000000000000000000000000000000000000000004c4b40',
  paymasterAndData: '0x',
  signature: '0x'
}
UserOp signed: {
  sender: '0x70997970C51812dc3A010C7d01b50e0d17dc79C8',
  nonce: 0n,
  initCode: '0x',
  callData: '0xb61d27f6000000000000000000000000948b3c65b89df0b4894abe91e6d02fe579834f8f000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000600000000000000000000000000000000000000000000000000000000000000004d09de08a00000000000000000000000000000000000000000000000000000000',
  accountGasLimits: '0x00000000000000000000000000000000000000000000000000000000000493e0',
  preVerificationGas: 50000n,
  gasFees: '0x00000000000000000000000000000000000000000000000000000000004c4b40',
  paymasterAndData: '0x',
  signature: '0x403e079bec12e08d822af12c0b7e4c9208b81efe39a5e247fe371e1b33e0dc5601e6e658f7f01cc7253448ed28bc315a6b1cf3f4aeb562778ac5dd91841b1b231c'
}
HttpRequestError: HTTP request failed.

Status: 500
URL: http://localhost:3000/rpc
Request body: {"method":"eth_sendUserOperation","params":[{"sender":"0x70997970C51812dc3A010C7d01b50e0d17dc79C8","nonce":"0x0","initCode":"0x","callData":"0xb61d27f6000000000000000000000000948b3c65b89df0b4894abe91e6d02fe579834f8f000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000600000000000000000000000000000000000000000000000000000000000000004d09de08a00000000000000000000000000000000000000000000000000000000","verificationGasLimit":"0x00000000000000000000000000000000000000000000000000000000000493e0","callGasLimit":"0x0000000000000000000000000000000000000000000000000000000000000000","maxFeePerGas":"0x00000000000000000000000000000000000000000000000000000000004c4b40","maxPriorityFeePerGas":"0x0000000000000000000000000000000000000000000000000000000000000000","preVerificationGas":"0xc350","paymasterAndData":"0x","signature":"0x403e079bec12e08d822af12c0b7e4c9208b81efe39a5e247fe371e1b33e0dc5601e6e658f7f01cc7253448ed28bc315a6b1cf3f4aeb562778ac5dd91841b1b231c"},"0x8464135c8F25Da09e49BC8782676a84730C318bC"]}

Details: {"message":"Cannot read properties of undefined (reading 'type')"}
Version: viem@2.44.4
    at Object.request (/home/mikitro/Desktop/Simple-AA-Wallet/node_modules/viem/utils/rpc/http.ts:158:17)
    at processTicksAndRejections (node:internal/process/task_queues:95:5)
    at async fn (/home/mikitro/Desktop/Simple-AA-Wallet/node_modules/viem/clients/transports/http.ts:148:19)
    at async request (/home/mikitro/Desktop/Simple-AA-Wallet/node_modules/viem/clients/transports/http.ts:153:39)
    at async delay.count.count (/home/mikitro/Desktop/Simple-AA-Wallet/node_modules/viem/utils/buildRequest.ts:150:22)
    at async attemptRetry (/home/mikitro/Desktop/Simple-AA-Wallet/node_modules/viem/utils/promise/withRetry.ts:44:22) {
  details: `{"message":"Cannot read properties of undefined (reading 'type')"}`,
  docsPath: undefined,
  metaMessages: [
    'Status: 500',
    'URL: http://localhost:3000/rpc',
    'Request body: {"method":"eth_sendUserOperation","params":[{"sender":"0x70997970C51812dc3A010C7d01b50e0d17dc79C8","nonce":"0x0","initCode":"0x","callData":"0xb61d27f6000000000000000000000000948b3c65b89df0b4894abe91e6d02fe579834f8f000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000600000000000000000000000000000000000000000000000000000000000000004d09de08a00000000000000000000000000000000000000000000000000000000","verificationGasLimit":"0x00000000000000000000000000000000000000000000000000000000000493e0","callGasLimit":"0x0000000000000000000000000000000000000000000000000000000000000000","maxFeePerGas":"0x00000000000000000000000000000000000000000000000000000000004c4b40","maxPriorityFeePerGas":"0x0000000000000000000000000000000000000000000000000000000000000000","preVerificationGas":"0xc350","paymasterAndData":"0x","signature":"0x403e079bec12e08d822af12c0b7e4c9208b81efe39a5e247fe371e1b33e0dc5601e6e658f7f01cc7253448ed28bc315a6b1cf3f4aeb562778ac5dd91841b1b231c"},"0x8464135c8F25Da09e49BC8782676a84730C318bC"]}'
  ],
  shortMessage: 'HTTP request failed.',
  version: '2.44.4',
  body: {
    method: 'eth_sendUserOperation',
    params: [ [Object], '0x8464135c8F25Da09e49BC8782676a84730C318bC' ]
  },
  headers: Headers {
    'content-type': 'application/json; charset=utf-8',
    'content-length': '99',
    date: 'Wed, 11 Mar 2026 10:52:48 GMT',
    connection: 'keep-alive',
    'keep-alive': 'timeout=72'
  },
  status: 500,
  url: 'http://localhost:3000/rpc'
}
```

Infine per completezza ho sistemato anche i file dentro `userop/runners/`

Che saranno comunque disponibili nella mia repo:
https://github.com/mikitro04/Simple-AA-Wallet